# Sensitivity Analysis — Greeks Heatmaps

This notebook computes the option **Greeks** (Delta and Vega) across a grid of
volatility (σ) and time-to-maturity (T) values using the Black-Scholes-Merton
analytical formula via finite differences, then renders interactive Plotly heatmaps.

| Greek | Finite-Difference Approx | Interpretation |
|-------|--------------------------|----------------|
| **Delta** (Δ) | $\partial C / \partial S_0$ | Sensitivity to spot-price change |
| **Vega** (ν) | $\partial C / \partial \sigma$ | Sensitivity to volatility change |

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.base_simulator import SimulatorConfig
from src.analytical import black_scholes

print('Imports OK')

## 1  Grid Definition

In [ ]:
# Fixed parameters
S0 = 100.0
K  = 100.0
r  = 0.05

# Grid axes
sigma_range = np.linspace(0.10, 0.50, 10)  # annualised volatility
T_range     = np.linspace(0.10, 2.00, 10)  # time to maturity (years)

print(f'σ grid: {sigma_range.round(3)}')
print(f'T grid: {T_range.round(3)}')

## 2  Compute Delta and Vega via Finite Differences

In [ ]:
def make_config(sigma: float, T: float) -> SimulatorConfig:
    """Create a throwaway SimulatorConfig for a (sigma, T) point."""
    return SimulatorConfig(
        S0=S0, K=K, T=T, r=r, sigma=sigma,
        n_paths=1000, n_steps=1, seed=None  # n_paths/n_steps irrelevant here
    )


def finite_delta(sigma: float, T: float, bump: float = 0.01) -> float:
    """Central-difference estimate of Delta = ∂C/∂S0."""
    cfg_up = SimulatorConfig(S0=S0 + bump, K=K, T=T, r=r, sigma=sigma, n_paths=1000, n_steps=1, seed=None)
    cfg_dn = SimulatorConfig(S0=S0 - bump, K=K, T=T, r=r, sigma=sigma, n_paths=1000, n_steps=1, seed=None)
    return (black_scholes(cfg_up, 'call') - black_scholes(cfg_dn, 'call')) / (2 * bump)


def finite_vega(sigma: float, T: float, bump: float = 0.001) -> float:
    """Central-difference estimate of Vega = ∂C/∂σ (× 0.01 → per 1% move)."""
    cfg_up = make_config(sigma + bump, T)
    cfg_dn = make_config(sigma - bump, T)
    return (black_scholes(cfg_up, 'call') - black_scholes(cfg_dn, 'call')) / (2 * bump) * 0.01


# Build 10×10 grids
delta_grid = np.array([[finite_delta(s, t) for t in T_range] for s in sigma_range])
vega_grid  = np.array([[finite_vega(s, t)  for t in T_range] for s in sigma_range])

print('Delta range:', delta_grid.min().round(4), '→', delta_grid.max().round(4))
print('Vega  range:', vega_grid.min().round(4),  '→', vega_grid.max().round(4))

## 3  Plot 4 — Interactive Delta Heatmap

In [ ]:
sigma_labels = [f'{s:.2f}' for s in sigma_range]
T_labels     = [f'{t:.2f}' for t in T_range]

fig_delta = go.Figure(data=go.Heatmap(
    z=delta_grid,
    x=T_labels,
    y=sigma_labels,
    colorscale='RdYlGn',
    zmid=0.5,
    colorbar=dict(title='Delta Δ'),
    hovertemplate='σ=%{y}<br>T=%{x}<br>Delta=%{z:.4f}<extra></extra>',
))

fig_delta.update_layout(
    title=dict(text='Option Delta (∂C/∂S₀) Across σ and T', font_size=15),
    xaxis_title='Time to Maturity T (years)',
    yaxis_title='Volatility σ',
    width=750, height=550,
    template='plotly_white',
)
fig_delta.show()

## 4  Plot 5 — Interactive Vega Heatmap

In [ ]:
fig_vega = go.Figure(data=go.Heatmap(
    z=vega_grid,
    x=T_labels,
    y=sigma_labels,
    colorscale='Blues',
    colorbar=dict(title='Vega ν (per 1% σ)'),
    hovertemplate='σ=%{y}<br>T=%{x}<br>Vega=%{z:.4f}<extra></extra>',
))

fig_vega.update_layout(
    title=dict(text='Option Vega (∂C/∂σ × 0.01) Across σ and T', font_size=15),
    xaxis_title='Time to Maturity T (years)',
    yaxis_title='Volatility σ',
    width=750, height=550,
    template='plotly_white',
)
fig_vega.show()

## 5  Side-by-Side Summary

In [ ]:
fig_both = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Delta Δ', 'Vega ν (per 1% σ)'],
    horizontal_spacing=0.12,
)

fig_both.add_trace(
    go.Heatmap(
        z=delta_grid, x=T_labels, y=sigma_labels,
        colorscale='RdYlGn', zmid=0.5,
        showscale=True, colorbar=dict(x=0.44, title='Δ'),
        hovertemplate='σ=%{y}<br>T=%{x}<br>Delta=%{z:.4f}<extra></extra>',
    ),
    row=1, col=1,
)
fig_both.add_trace(
    go.Heatmap(
        z=vega_grid, x=T_labels, y=sigma_labels,
        colorscale='Blues',
        showscale=True, colorbar=dict(x=1.0, title='ν'),
        hovertemplate='σ=%{y}<br>T=%{x}<br>Vega=%{z:.4f}<extra></extra>',
    ),
    row=1, col=2,
)

fig_both.update_xaxes(title_text='T (years)')
fig_both.update_yaxes(title_text='σ', col=1)
fig_both.update_layout(
    title='Greeks Sensitivity: Delta & Vega',
    width=1100, height=500,
    template='plotly_white',
)
fig_both.show()